![ab_testing_image](ab_testing_image.jpg)

As a Data Scientist at a leading online travel agency, you’ve been tasked with evaluating the impact of a new search ranking algorithm designed to improve conversion rates. The Product team is considering a full rollout, but only if the experiment shows a clear positive effect on the conversion rate and does not lead to a longer time to book.

They have shared A/B test datasets with session-level booking data (`"sessions_data.csv"`) and user-level control/variant split (`"users_data.csv"`). Your job is to analyze and interpret the results to determine whether the new ranking system delivers a statistically significant improvement and provide a clear, data-driven recommendation.

## `sessions_data.csv`

| column | data type | description | 
|--------|-----------|-------------|
| `session_id` | `string` | Unique session identifier (unique for each row) |
| `user_id` | `string` | Unique user identifier (non logged-in users have missing user_id values; each user can have multiple sessions) |
| `session_start_timestamp` | `string` | When a session started |
| `booking_timestamp` | `string` | When a booking was made (missing if no booking was made during a session) |
| `time_to_booking` | `float` | time from start of the session to booking, in minutes (missing if no booking was made during a session) |
| `conversion` | `integer` | _New column to create:_ did session end up with a booking (0 if booking_timestamp or time_to_booking is Null, otherwise 1) |

<br>

## `users_data.csv`

| column | data type | description | 
|--------|-----------|-------------|
| `user_id` | `string` | Unique user identifier (only logged-in users in this table) |
| `experiment_group` | `string` | control / variant split for the experiment (expected to be equal 50/50) |

<br>

The full on criteria are the following:
- Primary metric (conversion) effect must be statistically significant and show positive effect (increase).
- Guardrail (time_to_booking) effect must either be statistically insignificant or show positive effect (decrease)

In [104]:
import pandas as pd
from scipy.stats import chisquare
from pingouin import ttest
from statsmodels.stats.proportion import proportions_ztest

In [105]:
sessions = pd.read_csv('sessions_data.csv')
users = pd.read_csv('users_data.csv')

In [106]:
sessions.sample(5)

,session_id,user_id,session_start_timestamp,booking_timestamp,time_to_booking
10305,fzAKtRqUzyiKiZzY,GaUnok307imI2BdY,2025-01-03 09:19:00.074854851,NaN,NaN
16320,L8MheSULUdW5vWRM,sHYl871YiEqIefiY,2025-01-27 21:31:32.025309801,NaN,NaN
7846,MB717crZhuZyl6xb,FS0GmFOJkYRyUwZb,2025-01-06 19:19:32.456152678,NaN,NaN
8502,rZFgj9mtFewPPTjl,p22lpnChtgBjRqzI,2025-01-27 03:11:53.845743418,2025-01-27 03:25:58.907289546,14.084359
4392,Ab3CrqAogPJP8yBE,i7M062VfqfkPnRf7,2025-01-22 07:57:52.575376034,NaN,NaN


In [107]:
users.sample(5)

,user_id,experiment_group
4875,wCEnNQ4IG9ZeA2YX,control
8111,WSR3B6CzPGg3ZWnh,control
2792,CYtFRqrBcuYdykOt,variant
3269,IeARqWQs20QjFW8l,control
1304,4DMdPOnzOtLb1J0Z,control


### Your solution

In [108]:
confidence_level = 0.90  # Set the pre-defined confidence level (90%)
alpha = 1 - confidence_level  # Significance level for hypothesis tests

In [109]:
#  Creating the primary metric column
# Rule: conversion = 1 if a booking was made, 0 otherwise
# booking_timestamp is null when no booking happened
sessions['conversion'] = sessions['booking_timestamp'].notnull().astype(int)

# Quick check
print("Conversion value counts:")
print(sessions['conversion'].value_counts())

Conversion value counts:
0    14137
1     2844
Name: conversion, dtype: int64


In [110]:
#  Merge session-level data with user-level experiment group data
#  Using left join to keep all sessions, then drop ones we cant assign to a group
sessions_x_users = sessions.merge(users, on='user_id', how='left')

# Drop sessions from non-logged-in users (no experiment group assigned)
sessions_x_users = sessions_x_users.dropna(subset=['experiment_group'])

# Verify the merge
print(f"Total usable sessions : {len(sessions_x_users)}")
print(f"Shape                 : {sessions_x_users.shape}")
print("\nExperiment group counts:")
print(sessions_x_users['experiment_group'].value_counts())

Total usable sessions : 15283
Shape                 : (15283, 7)

Experiment group counts:
variant    7653
control    7630
Name: experiment_group, dtype: int64


In [111]:
# Sample Ratio Mismatch (SRM) test
# Check if the 50/50 split between control and variant is statistically valid
# Using Chi-Square test: if p-value > alpha, the split is fine

group_counts = sessions_x_users['experiment_group'].value_counts()
chi_stat, srm_chi2_pval = chisquare(group_counts)

print("SRM Test (Chi-Square):")
print(f"  Chi-Square statistic : {chi_stat:.4f}")
print(f"  srm_chi2_pval        : {srm_chi2_pval:.4f}")

if srm_chi2_pval > alpha:
    print(f"  Result: No SRM detected - split is valid (p={srm_chi2_pval:.4f} > alpha={alpha})")
else:
    print(f"  Result: SRM detected - split is biased (p={srm_chi2_pval:.4f} < alpha={alpha})")

SRM Test (Chi-Square):
  Chi-Square statistic : 0.0346
  srm_chi2_pval        : 0.8524
  Result: No SRM detected - split is valid (p=0.8524 > alpha=0.09999999999999998)


In [112]:
# Splitting the merged data into two separate groups for analysis
control_sessions = sessions_x_users[sessions_x_users['experiment_group'] == 'control']
variant_sessions = sessions_x_users[sessions_x_users['experiment_group'] == 'variant']

print(f"Control sessions : {len(control_sessions)}")
print(f"Variant sessions : {len(variant_sessions)}")

Control sessions : 7630
Variant sessions : 7653


In [113]:
# Effect analysis on the primary metric - conversion rate
# Using a one-tailed Z-test (we only care if variant is BETTER than control)
# H0: conversion rate is the same in both groups
# H1: variant has a HIGHER conversion rate than control

control_bookings = control_sessions['conversion'].sum()
variant_bookings = variant_sessions['conversion'].sum()
control_total = len(control_sessions)
variant_total = len(variant_sessions)

z_stat, p_value_conversion = proportions_ztest(
    count=[variant_bookings, control_bookings],
    nobs=[variant_total, control_total],
    alternative='larger'  # one-tailed: is variant larger than control?
)

print("Conversion Rate Z-test:")
print(f"  Control conversion rate  : {control_bookings/control_total:.4f}")
print(f"  Variant conversion rate  : {variant_bookings/variant_total:.4f}")
print(f"  Z-statistic              : {z_stat:.4f}")
print(f"  p_value_conversion       : {p_value_conversion:.4f}")

if p_value_conversion < alpha:
    print(f"  Result: SIGNIFICANT - variant converts better (p={p_value_conversion:.4f} < alpha={alpha})")
else:
    print(f"  Result: NOT significant (p={p_value_conversion:.4f} > alpha={alpha})")

Conversion Rate Z-test:
  Control conversion rate  : 0.1592
  Variant conversion rate  : 0.1819
  Z-statistic              : 3.7220
  p_value_conversion       : 0.0001
  Result: SIGNIFICANT - variant converts better (p=0.0001 < alpha=0.09999999999999998)


In [114]:
# Effect analysis on the guardrail metric - time to booking
# Using a two-tailed T-test (we care if time changes in EITHER direction)
# H0: average time to booking is the same in both groups
# H1: average time to booking is different between groups
# Only using sessions where a booking actually happened (conversion = 1)

control_time = control_sessions['time_to_booking'].dropna()
variant_time = variant_sessions['time_to_booking'].dropna()

ttest_result = ttest(variant_time, control_time, alternative='two-sided')
p_value_time = ttest_result['p-val'].values[0]

print("Time to Booking T-test:")
print(f"  Control avg time to booking  : {control_time.mean():.4f} mins")
print(f"  Variant avg time to booking  : {variant_time.mean():.4f} mins")
print(f"  T-statistic                  : {ttest_result['T'].values[0]:.4f}")
print(f"  p_value_time                 : {p_value_time:.4f}")

if p_value_time < alpha:
    print(f"  Result: SIGNIFICANT change in time to booking (p={p_value_time:.4f} < alpha={alpha})")
else:
    print(f"  Result: NOT significant - guardrail is safe (p={p_value_time:.4f} > alpha={alpha})")

Time to Booking T-test:
  Control avg time to booking  : 15.0124 mins
  Variant avg time to booking  : 14.8940 mins
  T-statistic                  : -0.6182
  p_value_time                 : 0.5365
  Result: NOT significant - guardrail is safe (p=0.5365 > alpha=0.09999999999999998)


In [115]:
# Step 6: Estimate effect sizes using ATE formula
# Formula: effect_size = avg(variant) / avg(control) - 1
# This gives the RELATIVE effect size (not absolute difference)

# Effect size for primary metric (conversion rate)
effect_size_primary = round(
    (variant_bookings/variant_total) / (control_bookings/control_total) - 1, 4
)

# Effect size for guardrail metric (time to booking)
effect_size_guardrail = round(
    variant_time.mean() / control_time.mean() - 1, 4
)

print(f"effect_size_primary   : {effect_size_primary:.4f}")
print(f"effect_size_guardrail : {effect_size_guardrail:.4f}")

effect_size_primary   : 0.1422
effect_size_guardrail : -0.0079


In [116]:
# Step 7: Making the final decision
# Criteria:
#   1. Conversion effect must be significant AND positive (increase)
#   2. Time to booking effect must be insignificant OR positive (decrease)
#
# Our results:
#   Conversion : p=0.0001 < alpha=0.10, effect=+0.1422 (significant & positive) -> PASS
#   Time       : p=0.5365 > alpha=0.10, effect=-0.0079 (not significant)        -> PASS
#
# Both criteria met -> Yes

decision_full_on = "Yes"
print(f"decision_full_on : {decision_full_on}")

decision_full_on : Yes


In [117]:
# Saving p-value for primary metric with the exact variable name grader expects
pval_primary = p_value_conversion

print(f"pval_primary : {pval_primary:.4f}")

pval_primary : 0.0001


In [118]:
# Saving p-value for guardrail metric with the exact variable name grader expects
pval_guardrail = p_value_time

print(f"pval_guardrail : {pval_guardrail:.4f}")

pval_guardrail : 0.5365


In [119]:
# Final verification of all variables the grader expects
srm_chi2_pval  = round(srm_chi2_pval, 4)
pval_primary   = round(p_value_conversion, 4)
pval_guardrail = round(p_value_time, 4)

print(f"srm_chi2_pval         : {srm_chi2_pval}")
print(f"pval_primary          : {pval_primary}")
print(f"pval_guardrail        : {pval_guardrail}")
print(f"effect_size_primary   : {effect_size_primary}")
print(f"effect_size_guardrail : {effect_size_guardrail}")
print(f"decision_full_on      : {decision_full_on}")

srm_chi2_pval         : 0.8524
pval_primary          : 0.0001
pval_guardrail        : 0.5365
effect_size_primary   : 0.1422
effect_size_guardrail : -0.0079
decision_full_on      : Yes
